# **Task 1a/b helper code**

In [6]:
import pandas as pd

# Load the CSV dataset
file_path = "HEA Phase DataSet v1d.csv"

df = pd.read_csv(file_path,encoding="cp1252")
display(df.head())

,Alloy ID,Alloy,Al,Co,Cr,Fe,Ni,Cu,Mn,Ti,...,Quenching,HPR,Microstructure_,Multiphase,IM_Structure,Microstructure,Phases,References,Unnamed: 51,Unnamed: 52
0,Alloy 0000,Al0.5NbTaTiV,0.111111,0.0,0.0,0.0,0.0,0.0,0.0,0.222222,...,NaN,NaN,BCC,S,NaN,BCC,BCC_SS,10.1016/j.proeng.2012.03.043,NaN,NaN
1,Alloy 0001,Al0.75MoNbTiV,0.157895,0.0,0.0,0.0,0.0,0.0,0.0,0.210526,...,NaN,NaN,BCC,S,NaN,BCC,BCC_SS,10.3390/e16020870,NaN,NaN
2,Alloy 0002,Al0.25MoNbTiV,0.058824,0.0,0.0,0.0,0.0,0.0,0.0,0.235294,...,NaN,NaN,BCC,S,NaN,BCC,BCC_SS,10.3390/e16020870,NaN,NaN
3,Alloy 0003,Al0.25NbTaTiV,0.058824,0.0,0.0,0.0,0.0,0.0,0.0,0.235294,...,NaN,NaN,BCC,S,NaN,BCC,BCC_SS,10.1016/j.proeng.2012.03.043,NaN,NaN
4,Alloy 0004,Al0.2MoTaTiV,0.047619,0.0,0.0,0.0,0.0,0.0,0.0,0.238095,...,NaN,NaN,BCC,S,NaN,BCC,BCC_SS,10.4028/www.scientific.net/MSF.898.638,NaN,NaN


In [15]:
import pandas as pd

# Load the CSV dataset
file_path = "LiIonDatabase.csv"

df = pd.read_csv(file_path, skiprows=4, header=0)
display(df.head())

,ID,composition,source,temperature,target,log_target,family,ChemicalFamily
0,1,Li2Ti0.75Ta0.25O3.125,10.1016/j.fusengdes.2010.04.003,300.00,9.000000e-05,-4.045757,Rocksalt,Oxides
1,2,Li2OHBr,10.1002/aenm.201700971,22.06,5.400000e-07,-6.267606,Anti-Perovskite,Oxides and Other Anions
2,3,Li2.4OH0.6Cl,10.1002/aenm.201700971,26.00,8.900000e-08,-7.050610,Anti-Perovskite,Oxides and Other Anions
3,4,Li2.1OH0.9Cl,10.1002/aenm.201700971,26.00,8.050000e-08,-7.094204,Anti-Perovskite,Oxides and Other Anions
4,5,Li2OHCl,10.1002/aenm.201700971,26.00,7.110000e-08,-7.148130,Anti-Perovskite,Oxides and Other Anions


In [8]:


import pandas as pd
import zipfile
import re
from pathlib import Path

zip_path = "Drilling data 2.zip"
samples = []

with zipfile.ZipFile(zip_path) as z:
    folders = sorted({p.split("/")[0] for p in z.namelist()})

    for folder in folders:

        # Keep only valid numbered CSV and TXT files
        csv_files = sorted(
            (p for p in z.namelist()
             if p.startswith(folder + "/")
             and re.fullmatch(r"auto\$\d{3}\.csv", Path(p).name)),
            key=lambda p: int(re.search(r"\$(\d+)", p).group(1))
        )

        txt_files = sorted(
            (p for p in z.namelist()
             if p.startswith(folder + "/")
             and re.fullmatch(r"\d+[bB]\.txt", Path(p).name)),
            key=lambda p: int(re.match(r"\d+", Path(p).name).group())
        )

        assert len(csv_files) == len(txt_files), folder

        # Simplify anomaly labels
        name = folder.lower()

        if "low pressure" in name:
            label = "Low pressure"
        elif "gap" in name:
            label = "Gap"
        elif "short" in name:
            label = "Short"
        elif "uneven tool" in name:
            label = "uneven tool"
        elif "anomaly" in name:
            label = "Anomaly"
        else:
            label = "Normal"

        for csv, txt in zip(csv_files, txt_files):

            # Sensor signals as NumPy array
            signal = pd.read_csv(
                z.open(csv), skiprows=[0, 1, 3]
            )[["INORM", "IEFF", "TCMD", "POSF"]].to_numpy()

            # Extract actual TXT measurements
            raw = z.read(txt).decode("latin1")
            values = [
                float(v) for v in re.findall(
                    r"Coord\.\s*Z\s*=\s*([-+]?\d+(?:\.\d+)?)",
                    raw
                )
            ]

            samples.append({
                "class": label,
                "csv": signal,
                **{f"Coord. Z {i}": v
                   for i, v in enumerate(values, 1)}
            })

data = pd.DataFrame(samples)

print(data.shape)
print(data["class"].value_counts())
display(data)


(305, 10)
class
Normal          79
Short           55
Low pressure    48
Gap             48
uneven tool     48
Anomaly         27
Name: count, dtype: int64


,class,csv,Coord. Z 1,Coord. Z 2,Coord. Z 3,Coord. Z 4,Coord. Z 5,Coord. Z 6,Coord. Z 7,Coord. Z 8
0,Anomaly,"[[-1.0, -9.1547146780592, 0.0, 0.0], [4.0, -9....",19.3675,19.4718,19.3305,19.4375,19.3337,19.4399,19.3769,19.4771
1,Anomaly,"[[-2.0, -9.169972535855964, 0.0, 0.0], [3.0, -...",19.2529,19.3414,19.2158,19.3146,19.2186,19.3139,19.2616,19.3473
2,Anomaly,"[[8.0, -9.1547146780592, 0.0, 0.0], [8.0, -9.1...",19.1292,19.1990,19.0907,19.1744,19.0970,19.1607,19.1343,19.2054
3,Anomaly,"[[-14.0, -9.231003967043026, 0.0, 0.0], [1.0, ...",19.9770,20.1003,19.9458,20.0803,19.9556,20.0764,19.9858,20.1027
4,Anomaly,"[[20.0, -9.139456820262437, 0.0, 0.0], [17.0, ...",19.9154,20.0308,19.8687,19.9899,19.8776,19.9931,19.9107,20.0204
...,...,...,...,...,...,...,...,...,...,...
300,Normal,"[[7.0, -9.35306682941715, 0.0, 0.0], [5.0, -9....",18.8647,18.8680,18.8757,18.8786,18.8701,18.8773,18.8650,18.8755
301,Normal,"[[6.0, -8.986878242294782, 0.0, 0.0], [5.0, -9...",18.8615,18.8643,18.8636,18.8793,18.8594,18.8645,18.8603,18.8625
302,Normal,"[[9.0, -8.92584681110772, 0.0, 0.0], [11.0, -9...",18.8577,18.8634,18.8609,18.8722,18.8683,18.8740,18.8668,18.8730
303,Normal,"[[5.0, -9.017393957888313, 0.0, 0.0], [6.0, -8...",18.8710,18.8804,18.8773,18.8873,18.8872,18.8853,18.8712,18.8784
